# Mostrar cartas:

Función que muestra una carta por pantalla, se usa para debug

In [123]:
from IPython.display import display, HTML

def display_card(url):
    # Ahora que tenemos la URL real, generamos el HTML
    html_code = f"""
    <div style="width: 300px; border-radius: 15px; overflow: hidden; box-shadow: 0 8px 16px rgba(0,0,0,0.3);">
        <img src="{url}" alt="Carta" style="width:100%; display: block;">
    </div>
    """
    display(HTML(html_code))

## Crear embeddings:

funcionalidad para crear embeddings apartir del texto y el nombre de una carta

In [124]:
# installar cv2
!pip install opencv-python

In [125]:
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import cv2


df = pd.read_csv("cards_final_with_xp.csv")

batch_size=32
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

try:
    texts = [f"{name} \n {text}" for name, text in zip(df["name"], df["text"])]
    query_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size).astype(np.float32)
finally:
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 405.96it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [126]:
query_embeddings.shape

(5346, 384)

# Embeddings a Redis
Función que recibe un embedding como un array de numpy y devuelve un blob que se le puede pasar a redis

In [127]:
import numpy as np

def to_blob(embedding: np.array) -> bytes:
    """
    Converts embedding to blob.
    :param embedding: embedding
    :return: blob
    """
    return embedding.astype(np.float32).tobytes()

# Iniciar conexión con redis

In [128]:
import redis

r = redis.Redis(host='redis', port=6379)

In [129]:
df

,code,name,text,type_code,traits,pack_code,illustrator,image_url,xp,faction_code
0,60401,Jacqueline Fine,[reaction] When an investigator at your locati...,investigator,Clairvoyant,jac,Aleksander Karcz,https://arkhamdb.com/bundles/cards/60401.jpg,0,mystic
1,60402,Arbiter of Fates,Jacqueline Fine deck only. [reaction] When you...,asset,Talent,jac,Pavel Kolomeyets,https://arkhamdb.com/bundles/cards/60402.jpg,0,mystic
2,60403,Dark Future,Revelation - Put Dark Future into play in your...,treachery,Omen|Endtimes,jac,Matt Bradbury,https://arkhamdb.com/bundles/cards/60403.jpg,0,neutral
3,60404,Nihilism,Revelation - Put Nihilism into play in your th...,treachery,Madness,jac,Sara Biddle,https://arkhamdb.com/bundles/cards/60404.jpg,0,neutral
4,60406,Scrying Mirror,Uses (4 secrets). [reaction] After a skill tes...,asset,Item|Charm,jac,Drazenka Kimpel,https://arkhamdb.com/bundles/cards/60406.jpg,0,mystic
...,...,...,...,...,...,...,...,...,...,...
5341,08129,Call for Backup,"One at a time, in any order, if you control a....",event,Favor|Synergy,eoep,Adam Schumpert,https://arkhamdb.com/bundles/cards/08129.jpg,2,neutral
5342,08130,Arm Injury,Revelation - Put Arm Injury into play in your ...,treachery,Injury,eoep,Robert Laskey,https://arkhamdb.com/bundles/cards/08130.jpg,0,neutral
5343,08131,Leg Injury,Revelation - Put Leg Injury into play in your ...,treachery,Injury,eoep,Robert Laskey,https://arkhamdb.com/bundles/cards/08131.jpg,0,neutral
5344,08132,Panic,Revelation - Put Panic into play in your threa...,treachery,Madness,eoep,Felicia Cano,https://arkhamdb.com/bundles/cards/08132.jpg,0,neutral


### Importar las cartas a redis

In [149]:
import pandas as pd
import redis

# 1. Conexión a Redis
# Ajusta host, port y db según tu configuración
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)


    # 2. Cargar el CSV con Pandas
df = pd.read_csv('cards_final_with_xp.csv')

print(df.dtypes)

df.drop_duplicates(inplace=True)

print(f"Iniciando la carga de {len(df)} registros...")

aux = 0
print(f'Hay {len(df)} filas en el DataFrame.')
# 3. Iterar sobre las filas y guardarlas en Redis
for _, row in df.iterrows():
    # Creamos una clave única para cada carta, por ejemplo "card:60401"
    card_key = row['code'] 
    if aux == 0:
        print(f"Ejemplo de clave: card:{card_key}")
        print(f"Ejemplo de datos: {row.to_dict()}")
        aux += 1
    
    # Convertimos la fila a un diccionario
    card_data = row.to_dict()
    
    # Usamos HSET para guardar todos los campos en un Hash de Redis
    r.hset(f"card:{card_key}", mapping=card_data)


# comprobar que se han importado todas las líneas del df
print(f"Se han importado {len(df)} registros.")

# ver el numero de cartas importadas
keys = r.keys("card:*")
print(f"Se han importado {len(keys)} cartas en Redis.")


print("¡Importación completada con éxito!")


code              str
name              str
text              str
type_code         str
traits            str
pack_code         str
illustrator       str
image_url         str
xp              int64
faction_code      str
dtype: object
Iniciando la carga de 5324 registros...
Hay 5324 filas en el DataFrame.
Ejemplo de clave: card:60401
Ejemplo de datos: {'code': '60401', 'name': 'Jacqueline Fine', 'text': '[reaction] When an investigator at your location would reveal any number of chaos tokens: Reveal 2 additional tokens. Of the revealed tokens, choose and cancel 2 non-[auto_fail] tokens, or 1 [auto_fail] token. (Limit once per round.) [elder_sign] effect: +1. If this effect is canceled or ignored, draw 1 card.', 'type_code': 'investigator', 'traits': 'Clairvoyant', 'pack_code': 'jac', 'illustrator': 'Aleksander Karcz', 'image_url': 'https://arkhamdb.com/bundles/cards/60401.jpg', 'xp': 0, 'faction_code': 'mystic'}
Se han importado 5324 registros.
Se han importado 5324 cartas en Redis.
¡Im

In [ ]:
print(df.xp.value_counts())

print("valores nulos en xp:", df.xp.isna().sum())


xp
0    4657
2     208
3     170
1     144
4      87
5      58
Name: count, dtype: int64
valores nulos en xp: 0
0


### Crear las funciones de las acciones

In [131]:
def card_exists(code):
    return r.exists(f"card:{code}") == 1

card_exists("60401")


True

In [132]:
def ver_carta_code(code, verbose=False):
    if card_exists(code):
        card_data = r.hgetall(f"card:{code}")
        for key, value in card_data.items():
            print(f"  {key}: {value}")

        if verbose:
            print(f"Datos de la carta {code}:")
            display_card(card_data["image_url"])

        return card_data
    
    print(f"No se encontró la carta con código {code}.")
    
    
        
        

ver_carta_code("60402", verbose=True)

  code: 60402
  type_code: asset
  illustrator: Pavel Kolomeyets
  name: Arbiter of Fates
  traits: Talent
  image_url: https://arkhamdb.com/bundles/cards/60402.jpg
  pack_code: jac
  xp: 0
  faction_code: mystic
  text: Jacqueline Fine deck only. [reaction] When you use Jacqueline Fine's [reaction] ability, exhaust Arbiter of Fates: This use of her ability does not count towards its limit.
Datos de la carta 60402:


{'code': '60402',
 'type_code': 'asset',
 'illustrator': 'Pavel Kolomeyets',
 'name': 'Arbiter of Fates',
 'traits': 'Talent',
 'image_url': 'https://arkhamdb.com/bundles/cards/60402.jpg',
 'pack_code': 'jac',
 'xp': '0',
 'faction_code': 'mystic',
 'text': "Jacqueline Fine deck only. [reaction] When you use Jacqueline Fine's [reaction] ability, exhaust Arbiter of Fates: This use of her ability does not count towards its limit."}

In [133]:
def meter_carta(code, name, text, type_code, traits, pack_code, faction_code, xp, illustrator, image_url):
    card_data = {
        "code": code,
        "name": name,
        "text": text,
        "type_code": type_code,
        "traits": traits,
        "pack_code": pack_code,
        "illustrator": illustrator,
        "image_url": image_url,
        "xp": xp,
        "faction_code": faction_code
    }
    r.hset(code, mapping=card_data)

meter_carta("99999", "Carta de Prueba", "Este es un texto de prueba.", "type_test", "trait1, trait2", "pack_test", "faction_test|faction_test2", 5, "Test Illustrator", "http://example.com/image.jpg")

ver_carta_code("99999")

No se encontró la carta con código 99999.


In [134]:
def eliminar_carta(code):
    if card_exists(code):
        r.delete(code)
        print(f"Carta con código {code} eliminada.")
    else:
        print(f"No se encontró la carta con código {code} para eliminar.")

eliminar_carta("99999")

ver_carta_code("99999")

No se encontró la carta con código 99999 para eliminar.
No se encontró la carta con código 99999.


# 2. INDICES

En  el  juego  hay  7  facciones,  5  que  son  clases  que  pueden  usar  los  jugadores  (mystic, 
survivor, guardian, seeker y rogue), la facción  “neutral” que es equipo común para todas 
las  clases  y  la  facción  “mythos”  que  son  los  enemigos.  Recientemente  se  han  añadido 
cartas  especiales  que  tienen  más  de  una  facción3.  Los  jugadores  están  interesados  en 
buscar todas las cartas que contengan varias facciones a la vez, así como, buscar todas las 
cartas que contengan al menos una de las facciones que indiquen. Por defecto devolvemos 
las cartas de 5 en 5. 

In [135]:
import redis
from redis.commands.search.field import TextField, TagField, NumericField
from redis.commands.search.index_definition import IndexDefinition, IndexType

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 1. Definimos el esquema (qué columnas queremos indexar)

schema = (
            NumericField("code"),           # NUMERIC para búsquedas exactas por código
            TextField("name"),  # TEXT para búsquedas de palabras (el nombre pesa más 
            TextField("text"),              # TEXT para el cuerpo de la carta
            TextField("type_code"),         # TEXT para el tipo de carta
            TagField("traits", separator="|"), # TAG indicando que los rasgos se separan por '|'
            TagField("pack_code"),          # TAG para el paquete al que pertenece la carta
            TagField("faction_code", separator="|"),       # TAG para la facción a la que pertenece la carta
            NumericField("xp"),          # NUMERIC para filtros de rango (ej: xp > 2)
            TextField("illustrator"),       # TEXT para el ilustrador de la carta
            TextField("image_url")        # TEXT para la URL de la imagen de la carta  
        )

try:
    r.ft("cartas:").create_index(
        schema,
        
        definition=IndexDefinition(
            prefix=["card:"],       # SOLO indexará llaves que empiecen por "card:"
            index_type=IndexType.HASH
        )
    )
    print("✅ Índice creado con éxito")
except Exception as e:
    print(f"❌ Error o el índice ya existe: {e}")

✅ Índice creado con éxito


In [136]:
def buscar_cartas_varias_facciones():
    
    # Usamos r"" para que Python no toque las barras invertidas
    # Escapamos el pipe con \ para que RediSearch lo tome literal y no como un "OR"
    query = r"@faction_code:*\|*"

    # Importante: FT.SEARCH espera el índice y luego la query como argumentos separados
    response = r.execute_command("FT.SEARCH", "cartas:", query)

    return response

# Ejecución
print(buscar_cartas_varias_facciones())


[0]


In [137]:
from redis.commands.search.aggregation import AggregateRequest, Reducer

def buscar_cartas_faccion(*factions, page_size=5, page_number=0):
    # Usamos FT.AGGREGATE para procesar los datos
    # 1. Cargamos el campo faction_code
    # 2. Contamos cuántos elementos hay en la lista de etiquetas (tags)
    # 3. Filtramos los que tengan más de 1
    
    offset = page_number * page_size

    # 1. Iniciamos la petición cargando los campos necesarios
    request = AggregateRequest("*").load("@code", "@faction_code")

    if factions:
        
        condiciones = [f'contains(@faction_code, "{f}")' for f in factions]

        request.filter("&&".join(condiciones))
        
    else:
        request.filter('contains(@faction_code, "|")')

    request.limit(offset, page_size)

    response = r.ft("cartas:").aggregate(request)

    solucion = response.rows
    return solucion

# Ejecución
resultado = buscar_cartas_faccion()
resultado

[['code', '5116', 'faction_code', 'seeker|mystic']]

In [138]:
# cambiar a redis 7.1.1

!pip install redis==7.1.1

In [139]:
redis.__version__

'7.2.0'

In [140]:
a = "hola|mundo"
b = "hola"

b.split("|")

['hola']

In [145]:
len(df)

5324

In [ ]:
from redis.commands.search.reducers import count

def sacar_traits_mas_usados():
    request = (
        AggregateRequest("*")
        .load("@code", "@traits")
        
    )

    

    query = r.ft("cartas:").aggregate(request).rows
    diccionario = {}
    
    print(len(query))

    for item in query:
        traits = item[3].split("|")
        for trait in traits:
            if trait in diccionario:
                diccionario[trait] += 1
            else:
                diccionario[trait] = 1

    diccionario = dict(sorted(diccionario.items(), key=lambda item: item[1], reverse=True))

    return diccionario

trait = sacar_traits_mas_usados()
trait


4734


{'No traits': 880,
 'Item': 458,
 'Monster': 275,
 'Ally': 208,
 'Humanoid': 206,
 'Elite': 187,
 'Spell': 150,
 'Hazard': 122,
 'Relic': 122,
 'Terror': 115,
 'Talent': 108,
 'Weapon': 107,
 'Insight': 105,
 'Tactic': 93,
 'Cursed': 87,
 'Ancient': 81,
 'Spirit': 80,
 'Blessed': 78,
 'Curse': 76,
 'Miskatonic': 75,
 'Power': 73,
 'Arkham': 65,
 'Tome': 65,
 'Creature': 64,
 'Innate': 64,
 'Cultist': 63,
 'Illicit': 63,
 'Trick': 62,
 'Hex': 60,
 'Scheme': 60,
 "R'lyeh": 59,
 'Ruins': 58,
 'Melee': 58,
 'Cave': 57,
 'Tool': 57,
 'Ritual': 56,
 'Charm': 55,
 'Woods': 51,
 'Dunwich': 50,
 'Otherworld': 49,
 'Science': 48,
 'Firearm': 46,
 'Fortune': 45,
 'Pact': 44,
 'Criminal': 43,
 'Extradimensional': 43,
 'Practiced': 43,
 'Omen': 40,
 'Madness': 38,
 'Spectral': 35,
 'Wayfarer': 32,
 'Glyph': 31,
 'Innsmouth': 28,
 'Abomination': 27,
 'Gambit': 27,
 'Ancient One': 26,
 'Mutated': 24,
 'Silver Twilight': 23,
 'City': 23,
 'Guest': 21,
 'Sorcerer': 21,
 'Eidolon': 20,
 'Sanctum': 20,
 

In [142]:
df.traits.value_counts()

traits
No traits                      1052
Hazard                          110
Terror                          102
Spell                            93
Talent                           80
                               ... 
Trick|Synergy                     1
Ritual|Synergy                    1
Innate|Synergy                    1
Spell|Blessed|Weapon|Ranged       1
Favor|Synergy                     1
Name: count, Length: 961, dtype: int64

In [143]:
aaaaa

NameError: name 'aaaaa' is not defined

In [ ]:
r.flushall()

True